In [2]:

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "MIG-e6b94c71-1db7-5ad2-a95a-9de8bd50bb1a"
# ----------------------------------------------------------------------------

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import h5py
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
import time
from tqdm.auto import tqdm


LEARNING_RATE_AE = 0.0005
LEARNING_RATE_MLP = 0.0005
BATCH_SIZE = 32  
NUM_EPOCHS_AE = 15 
NUM_EPOCHS_MLP = 30 
NUM_CLASSES = 12
LATENT_DIM = 128 # The size of compressed "fingerprint"

TRAIN_DATA_FILE = 'train.h5'
TEST_DATA_FILE = 'test.h5'
TEST_LABELS_FILE = 'test_labels.csv' 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def load_and_preprocess_data():
    """Loads, concatenates, transposes, and scales all data."""
    print("Loading labels...")
    with h5py.File(TRAIN_DATA_FILE, 'r') as hdf:
        y_train = np.array(hdf.get('Class/1'))
    y_test = pd.read_csv(TEST_LABELS_FILE, header=None).values.squeeze()

    print("Loading and concatenating training spectra...")
    all_train_spectra = []
    with h5py.File(TRAIN_DATA_FILE, 'r') as hdf:
        train_keys = [f"{i:03d}" for i in range(1, 101)]
        for key in tqdm(train_keys, desc="Training data"):
            data_block = np.array(hdf[f'Spectra/{key}'])
            all_train_spectra.append(data_block.T)
    X_train_raw = np.concatenate(all_train_spectra, axis=0)

    print("Loading and concatenating test spectra...")
    all_test_spectra = []
    with h5py.File(TEST_DATA_FILE, 'r') as hdf:
        test_keys = ['1', '2']
        for key in tqdm(test_keys, desc="Test data    "):
            data_block = np.array(hdf[f'UNKNOWN/{key}'])
            all_test_spectra.append(data_block.T)
    X_test_raw = np.concatenate(all_test_spectra, axis=0)

    print("Scaling data (this can take several minutes)...")
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_test_scaled = scaler.transform(X_test_raw)
    
    print("Data loading and preprocessing complete.")
    return X_train_scaled, y_train, X_test_scaled, y_test



class Autoencoder(nn.Module):
    def __init__(self, input_dim=40002, latent_dim=LATENT_DIM):
        super(Autoencoder, self).__init__()
        
        """ Encoder: 40002 -> 2048 -> 512 -> 128"""
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 2048),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, latent_dim) # The "fingerprint"
        )
        
        """ Decoder: 128 -> 512 -> 2048 -> 40002"""
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 2048),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(2048, input_dim) # Reconstruct the original
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded



class MLPClassifier(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, num_classes=NUM_CLASSES):
        super(MLPClassifier, self).__init__()
        self.layer = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        return self.layer(x)


class LibsDatasetAE(Dataset):
    """Dataset for Autoencoder: returns the sample as both input and target."""
    def __init__(self, data):
        # AE uses Linear layers, so we don't add a channel dim
        # We need (N, 40002)
        self.X = torch.tensor(data, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        # For AE, input and target are the same
        return self.X[idx], self.X[idx]

class LibsDatasetClassify(Dataset):
    """Dataset for Classifier: returns the LATENT vector and the label."""
    def __init__(self, data, labels, trained_encoder):
        # We pass the data through the trained, frozen encoder
        trained_encoder.eval()
        with torch.no_grad():
            self.X_latent = trained_encoder(torch.tensor(data, dtype=torch.float32).to(device)).cpu()
        
        self.y = torch.tensor(labels, dtype=torch.long)
        # Labels are 1-indexed (1-12), convert to 0-indexed (0-11)
        self.y = self.y - 1 

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        # Return the pre-computed latent vector and its label
        return self.X_latent[idx], self.y[idx]


def train_autoencoder(model, train_loader, criterion, optimizer, epoch):
    model.train()
    total_loss = 0
    start_time = time.time()
    
    for i, (inputs, targets) in enumerate(tqdm(train_loader, desc=f"AE Epoch {epoch+1}/{NUM_EPOCHS_AE}")):
        inputs = inputs.to(device)
        targets = targets.to(device)
        
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
            
    avg_loss = total_loss / len(train_loader)
    epoch_time = time.time() - start_time
    print(f'--- AE Epoch {epoch+1} Summary ---')
    print(f'Average Training Loss (MSE): {avg_loss:.4f}, Time: {epoch_time:.2f}s')

def train_model(model, train_loader, criterion, optimizer, epoch):
    model.train()
    total_loss = 0
    start_time = time.time()
    
    for i, (spectra, labels) in enumerate(tqdm(train_loader, desc=f"MLP Epoch {epoch+1}/{NUM_EPOCHS_MLP}")):
        spectra = spectra.to(device)
        labels = labels.to(device)
        
        outputs = model(spectra)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
            
    avg_loss = total_loss / len(train_loader)
    epoch_time = time.time() - start_time
    print(f'--- MLP Epoch {epoch+1} Summary ---')
    print(f'Average Training Loss: {avg_loss:.4f}, Time: {epoch_time:.2f}s')

def evaluate_model(model, test_loader, criterion):
    # This function is used only for the final MLP classifier
    print("\nEvaluating MLP Classifier on test data...")
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0
    
    with torch.no_grad():
        for spectra, labels in tqdm(test_loader, desc="Evaluating MLP"):
            spectra = spectra.to(device)
            labels = labels.to(device)
            
            outputs = model(spectra)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            
            _, predicted = torch.max(outputs.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(test_loader)
    accuracy = accuracy_score(all_labels, all_preds)
    print(f'Test Loss: {avg_loss:.4f}')
    print(f'Test Accuracy: {accuracy * 100:.2f}%')
    print("\n--- Classification Report ---")
    target_names = [f"Class {i+1}" for i in range(NUM_CLASSES)]
    print(classification_report(all_labels, all_preds, labels=list(range(NUM_CLASSES)), target_names=target_names, digits=4, zero_division=0))

#  Main Execution ---

if __name__ == "__main__":
    
    print(f"Using device: {device}")
    
    # --- Data ---
    X_train, y_train, X_test, y_test = load_and_preprocess_data()
    
    # =========================================================================
    # PHASE 1: TRAIN THE AUTOENCODER
    # =========================================================================
    
    print("\n--- Starting PHASE 1: Autoencoder Training ---")
    
    ae_dataset = LibsDatasetAE(X_train)
    ae_loader = DataLoader(dataset=ae_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
    
    autoencoder_model = Autoencoder(input_dim=X_train.shape[1]).to(device)
    criterion_ae = nn.MSELoss()
    optimizer_ae = optim.Adam(autoencoder_model.parameters(), lr=LEARNING_RATE_AE)
    
    print(f"Autoencoder model created with {LATENT_DIM} latent features.")
    
    for epoch in range(NUM_EPOCHS_AE):
        train_autoencoder(autoencoder_model, ae_loader, criterion_ae, optimizer_ae, epoch)
        
    print("--- Autoencoder Training Finished ---")

    # =========================================================================
    # PHASE 2: TRAIN THE CLASSIFIER
    # =========================================================================
    
    print("\n--- Starting PHASE 2: MLP Classifier Training ---")
    
    # Get the trained encoder part
    trained_encoder = autoencoder_model.encoder
    trained_encoder.eval() # Freeze it

    print("\nCalculating class weights for weighted loss...")
    class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
    print(f"Weights calculated: {class_weights_tensor}")

    print("\nCreating new datasets from latent features...")
    # This dataset passes all data through the encoder once (fast)
    train_dataset_mlp = LibsDatasetClassify(X_train, y_train, trained_encoder)
    test_dataset_mlp = LibsDatasetClassify(X_test, y_test, trained_encoder)
    
    train_loader_mlp = DataLoader(dataset=train_dataset_mlp, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
    test_loader_mlp = DataLoader(dataset=test_dataset_mlp, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
    
    # --- Model ---
    mlp_model = MLPClassifier(latent_dim=LATENT_DIM).to(device)
    print(f"MLP Classifier model created.")

    # --- Loss, Optimizer, Scheduler ---
    criterion_mlp = nn.CrossEntropyLoss(weight=class_weights_tensor)
    optimizer_mlp = optim.Adam(mlp_model.parameters(), lr=LEARNING_RATE_MLP)
    scheduler_mlp = torch.optim.lr_scheduler.StepLR(optimizer_mlp, step_size=15, gamma=0.1)
    print("Using Weighted Loss, Adam Optimizer, and StepLR Scheduler.")

    # --- Training ---
    print(f"\nStarting MLP training for {NUM_EPOCHS_MLP} epochs...")
    total_start_time = time.time()
    
    for epoch in range(NUM_EPOCHS_MLP):
        train_model(mlp_model, train_loader_mlp, criterion_mlp, optimizer_mlp, epoch)
        scheduler_mlp.step()
        
    total_end_time = time.time()
    print(f"\n--- MLP Training Finished ---")
    print(f"Total MLP training time: {(total_end_time - total_start_time)/60:.2f} minutes")

    # --- Evaluation ---
    evaluate_model(mlp_model, test_loader_mlp, criterion_mlp)

Using device: cuda
Loading labels...
Loading and concatenating training spectra...


Training data:   0%|          | 0/100 [00:00<?, ?it/s]

Loading and concatenating test spectra...


Test data    :   0%|          | 0/2 [00:00<?, ?it/s]

Scaling data (this can take several minutes)...
Data loading and preprocessing complete.

--- Starting PHASE 1: Autoencoder Training ---
Autoencoder model created with 128 latent features.


...

--- MLP Epoch 29 Summary ---
Average Training Loss: 1.4016, Time: 11.05s


MLP Epoch 30/30:   0%|          | 0/1563 [00:00<?, ?it/s]

--- MLP Epoch 30 Summary ---
Average Training Loss: 1.4034, Time: 11.11s

--- MLP Training Finished ---
Total MLP training time: 5.24 minutes

Evaluating MLP Classifier on test data...


Evaluating MLP:   0%|          | 0/625 [00:00<?, ?it/s]

Test Loss: 3.6961
Test Accuracy: 27.67%

--- Classification Report ---
              precision    recall  f1-score   support

     Class 1     0.1648    0.0884    0.1150      3146
     Class 2     0.8832    0.2464    0.3853      3129
     Class 3     0.0078    0.0055    0.0065       543
     Class 4     0.3571    0.1996    0.2561      1603
     Class 5     0.1014    0.1336    0.1153      1048
     Class 6     0.4311    0.9817    0.5991      1589
     Class 7     0.0124    0.0571    0.0204       560
     Class 8     0.3766    0.3642    0.3703      1546
     Class 9     0.8824    0.3959    0.5466      3127
    Class 10     0.1400    0.9300    0.2434       514
    Class 11     0.4935    0.0473    0.0863      3195
    Class 12     0.0000    0.0000    0.0000         0

    accuracy                         0.2767     20000
   macro avg     0.3208    0.2875    0.2287     20000
weighted avg     0.4823    0.2767    0.2874     20000



In [1]:
import torch
print(f"Is CUDA available in this notebook? {torch.cuda.is_available()}")

Is CUDA available in this notebook? True
